# Search-03g · Mémoire bornée : ce qu'un plafond de frontière coûte

Ce carnet est une **accrétion de [Search-03 (recherche informée et A\*)](Search-03-Informed.ipynb)**. Search-03 mesure le **travail** d'un algorithme : son compteur est l'**expansion**. Il parle de mémoire en prose — `O(bd)` pour IDA\*, « mémoire limitée » dans son tableau de choix — mais il ne la **compte jamais**.

Ce carnet prend l'autre axe et le mesure :

| Question | Ce que ce carnet compte |
|---|---|
| Combien de mémoire A\* occupe-t-il vraiment ? | le **pic de frontière** (états distincts en attente) **et** les **états atteints** (le dictionnaire jamais vidé) |
| IDA\* tient-il sa promesse de mémoire linéaire ? | le **pic de pile** (longueur du chemin courant) |
| Que fait un plafond de frontière à la garantie d'optimalité ? | le **coût rendu** pour un budget donné, face à l'optimum |

Le terrain est le **taquin 8** avec l'heuristique de Manhattan — un domaine où l'on peut comparer trois stratégies sur **exactement les mêmes instances**, avec des graines fixées.

**Navigation** : [<< Search-03f — réparer localement sous garantie (LPA\*)](Search-03f-Reparer-Localement-Sous-Garantie-Python.ipynb) | [Index](README.md) | [retour au parent Search-03 (A\*) ↑](Search-03-Informed.ipynb)

## 1. Le terrain : taquin 8, heuristique de Manhattan

L'état est un tuple de neuf entiers, `0` désignant la case vide. Le but est l'ordre naturel. Les instances sont **engendrées depuis le but** par une marche aléatoire à graine fixée : elles sont donc solubles par construction, et reproductibles d'une exécution à l'autre.

La distance de Manhattan est **admissible** (chaque tuile est au moins aussi loin que sa distance en lignes et colonnes) et **consistante** sur ce domaine — c'est ce qui rend les trois algorithmes comparables : aucun ne perd l'optimalité par la faute de l'heuristique.

In [1]:
# Taquin 8 : etat = tuple de 9 entiers, 0 = case vide. Le but est l'ordre naturel.
import heapq
import itertools
import random

BUT = (1, 2, 3, 4, 5, 6, 7, 8, 0)
INFINI = float("inf")

VOISINS = {
    0: (1, 3), 1: (0, 2, 4), 2: (1, 5),
    3: (0, 4, 6), 4: (1, 3, 5, 7), 5: (2, 4, 8),
    6: (3, 7), 7: (4, 6, 8), 8: (5, 7),
}


def manhattan(etat):
    """Somme des distances de Manhattan de chaque tuile a sa place au but."""
    return sum(abs(i // 3 - (v - 1) // 3) + abs(i % 3 - (v - 1) % 3)
               for i, v in enumerate(etat) if v)


def successeurs(etat):
    """Les etats atteignables en glissant une tuile dans la case vide."""
    vide = etat.index(0)
    for cible in VOISINS[vide]:
        t = list(etat)
        t[vide], t[cible] = t[cible], t[vide]
        yield tuple(t)


def melanger(coups, graine):
    """Marche aleatoire depuis le but : instance soluble et reproductible."""
    rnd = random.Random(graine)
    etat, precedent = BUT, None
    for _ in range(coups):
        vide = etat.index(0)
        cible = rnd.choice([c for c in VOISINS[vide] if c != precedent])
        t = list(etat)
        t[vide], t[cible] = t[cible], t[vide]
        etat, precedent = tuple(t), vide
    return etat

### Des instances reproductibles

Chaque instance est obtenue par une marche aléatoire de 20 coups depuis le but : elle est donc soluble par construction, et la graine fixe l'instance. On mesure d'abord la borne inférieure `h` de chacune. C'est elle qui fixe le coût que l'heuristique annonce — et l'on verra qu'elle ne dit rien de la mémoire que la recherche devra dépenser.


In [2]:
INSTANCES = {graine: melanger(20, graine) for graine in (1, 2, 3)}

print("instance | h | etat")
print("-" * 46)
for graine, etat in INSTANCES.items():
    print(f"graine {graine} | h = {manhattan(etat):2d} | {etat}")
    for ligne in range(3):
        print("   ", etat[ligne * 3:ligne * 3 + 3])
    print()

instance | h | etat
----------------------------------------------
graine 1 | h = 20 | (7, 5, 4, 8, 0, 1, 6, 3, 2)
    (7, 5, 4)
    (8, 0, 1)
    (6, 3, 2)

graine 2 | h = 10 | (1, 3, 0, 7, 6, 5, 8, 2, 4)
    (1, 3, 0)
    (7, 6, 5)
    (8, 2, 4)

graine 3 | h = 12 | (8, 1, 3, 7, 0, 5, 4, 6, 2)
    (8, 1, 3)
    (7, 0, 5)
    (4, 6, 2)



### Lecture du résultat

Les trois instances n'ont pas la même difficulté, et `h` — la borne inférieure du coût optimal — ne la prédit pas : **20, 10, puis 12**. L'instance de la graine 3 a le `h` le plus faible des trois et c'est pourtant elle qui demandera le plus de mémoire : la section suivante mesure exactement cet écart entre ce que l'heuristique annonce et ce que la frontière coûte.

## 2. A\* : ce que la frontière ne dit pas

A\* en **graphe** garde les états générés non encore dépilés dans sa frontière — mais il garde aussi, jusqu'à la fin, le dictionnaire `g` de **tous** les états atteints. On compte donc trois choses distinctes :

- les **expansions** : les états effectivement finalisés (dépilés une seule fois, avec leur `g` définitif) ;
- le **pic de frontière** : le plus grand nombre d'états distincts simultanément en attente ;
- les **états atteints** : la taille du dictionnaire `g`, qui ne se vide jamais — même frontière vide, même solution trouvée.

La troisième est celle qu'on oublie. Le pic de frontière décrit la **pression d'un moment** ; les états atteints décrivent ce que le processus retient **à la fin**. La mémoire réelle d'un A\* en graphe est la somme des deux, et la mesure ci-dessous montre laquelle domine.

L'`open_set` est tenu à jour à part du tas, parce qu'un tas peut contenir plusieurs entrées périmées pour le même état : compter `len(tas)` mesurerait les doublons, pas la mémoire.

In [3]:
def astar(depart):
    """A* en graphe. Rend (cout, expansions, pic de frontiere, etats, chemin).

    `etats` est le nombre d'etats atteints, c'est-a-dire la taille du
    dictionnaire `g` : il ne se vide jamais, meme frontiere vide.
    """
    g = {depart: 0}
    venu_de = {}
    compteur = itertools.count()
    tas = [(manhattan(depart), next(compteur), depart)]
    open_set = {depart}
    expansions = pic = 1
    while tas:
        _, _, courant = heapq.heappop(tas)
        if courant not in open_set:      # entree perimee : l'etat a deja ete finalise
            continue
        open_set.discard(courant)
        expansions += 1
        if courant == BUT:
            cout = g[courant]
            chemin = [courant]
            while courant in venu_de:
                courant = venu_de[courant]
                chemin.append(courant)
            return cout, expansions, pic, len(g), chemin[::-1]
        for suivant in successeurs(courant):
            ng = g[courant] + 1
            if ng < g.get(suivant, INFINI):
                g[suivant] = ng
                venu_de[suivant] = courant
                heapq.heappush(tas, (ng + manhattan(suivant), next(compteur), suivant))
                open_set.add(suivant)
                pic = max(pic, len(open_set))
    return INFINI, expansions, pic, len(g), None


resultats_astar = {graine: astar(etat) for graine, etat in INSTANCES.items()}

print(f"{'graine':>6} | {'cout':>4} | {'expansions':>10} | {'pic frontier':>12} | {'etats atteints':>14} | {'ratio':>5}")
print("-" * 64)
for graine, (cout, expansions, pic, etats, _) in resultats_astar.items():
    print(f"{graine:>6} | {cout:>4} | {expansions:>10} | {pic:>12} | {etats:>14} | {etats / pic:>5.2f}")

graine | cout | expansions | pic frontier | etats atteints | ratio
----------------------------------------------------------------
     1 |   20 |         51 |           28 |             77 |  2.75
     2 |   14 |         51 |           39 |             88 |  2.26
     3 |   18 |        274 |          170 |            442 |  2.60


### Lecture du résultat

Les trois instances demandent le **même ordre de grandeur de travail** (51 à 274 expansions) mais des mémoires qui varient d'un facteur **6** : 28, 39, puis **170** états simultanément en attente pour la graine 3. Le pic n'est pas une constante du domaine — c'est une propriété de l'instance.

La dernière colonne corrige une lecture trop charitable du pic : les **états atteints** valent 77, 88 et **442**, soit 2,3 à 2,8 fois le pic de frontière sur les trois instances. Le dictionnaire `g` ne se vide jamais, donc la mémoire d'un A\* en graphe est **dominée par lui**, pas par la frontière. Un carnet qui ne compterait que le pic annoncerait 170 là où le processus en retient 442 : c'est précisément l'erreur que cette colonne existe pour rendre visible.

C'est la raison pour laquelle « A\* tient en mémoire » n'est pas une phrase qu'on peut écrire sans mesurer : sur ce taquin de neuf cases, la frontière d'une instance facile tient dans quelques dizaines d'états, celle d'une instance plus tordue en demande dix fois plus — et le dictionnaire des états atteints, lui, reste en mémoire jusqu'au bout.

## 3. IDA\* : la mémoire devient la profondeur

IDA\* n'empile jamais la frontière. Il descend en profondeur sous une borne croissante sur `f = g + h`, et ne garde donc que le **chemin courant**. Sa mémoire est `O(d)` où `d` est la profondeur de la solution, pas le nombre d'états en attente.

Le prix se paie ailleurs : entre deux bornes, les mêmes états sont **revisités**. Le compteur ci-dessous compte les **entrées** dans le parcours en profondeur, toutes itérations confondues — ce n'est pas le même compteur que les expansions d'A\*, qui compte des états finalisés. Les deux ne se comparent donc pas directement ; ce qui se compare, c'est la mémoire.

In [4]:
def ida_etoile(depart):
    """IDA*. Rend (cout, entrees, pic de pile, chemin).

    `entrees` compte chaque visite de noeud, toutes bornes confondues :
    a ne pas confondre avec les expansions (etats finalises) d'A*.
    """
    borne = manhattan(depart)
    chemin = [depart]
    entrees = 0
    pic = 1
    solution = []

    def descendre(g, borne):
        nonlocal entrees, pic
        noeud = chemin[-1]
        f = g + manhattan(noeud)
        if f > borne:
            return f
        entrees += 1
        pic = max(pic, len(chemin))
        if noeud == BUT:
            solution[:] = list(chemin)
            return -1
        meilleur = INFINI
        for suivant in successeurs(noeud):
            if suivant in chemin:        # pas de retour en arriere dans le chemin courant
                continue
            chemin.append(suivant)
            t = descendre(g + 1, borne)
            chemin.pop()
            if t == -1:
                return -1
            if t < meilleur:
                meilleur = t
        return meilleur

    while True:
        t = descendre(0, borne)
        if t == -1:
            return len(solution) - 1, entrees, pic, list(solution)
        if t == INFINI:
            return INFINI, entrees, pic, None
        borne = t


resultats_ida = {graine: ida_etoile(etat) for graine, etat in INSTANCES.items()}

print(f"{'graine':>6} | {'cout':>4} | {'entrees':>8} | {'pic de pile':>11}")
print("-" * 36)
for graine, (cout, entrees, pic, _) in resultats_ida.items():
    print(f"{graine:>6} | {cout:>4} | {entrees:>8} | {pic:>11}")

graine | cout |  entrees | pic de pile
------------------------------------
     1 |   20 |       21 |          21
     2 |   14 |       65 |          15
     3 |   18 |      188 |          19


### Lecture du résultat

Le **coût est identique** à celui d'A\* sur les trois instances (20, 14, 18) : IDA\* n'a rien relâché. Le **pic de pile** vaut 21, 15 et 19 — il suit la longueur du chemin, pas la difficulté de l'instance. Sur la graine 3, A\* retenait 442 états atteints pour un pic de frontière de 170, IDA\* en tient 19 : **vingt-trois fois moins** que ce qu'A\* garde, pour le même résultat optimal.

Le compteur d'entrées, lui, n'est pas comparable terme à terme avec les expansions : sur ces instances il tombe entre 21 et 188, tantôt sous les expansions d'A\*, tantôt au-dessus. C'est normal — un état peut être finalisé une fois par A\* et visité plusieurs fois par IDA\* à mesure que la borne monte, et les deux compteurs n'attribuent pas la même chose au même événement. Retenez la **mémoire**, pas le nombre brut.

## 4. Le raccourci tentant : plafonner la frontière

Si A\* coûte trop de mémoire, l'idée la plus naturelle est de **plafonner sa frontière** : dès qu'elle dépasse un budget `B`, on jette l'état de plus grand `f`. Le tas devient borné, l'algorithme continue de tourner.

C'est ce que fait `astar_frontiere_bornee` ci-dessous. Trois détails comptent pour la suite :

- le tas peut encore contenir des entrées périmées ; c'est l'`open_set` qui porte le plafond ;
- un état jeté n'est **pas** interdit : s'il est régénéré plus tard avec un `g` meilleur, il revient. Le plafond ne perd donc pas la solution, il perd l'ordre dans lequel A\* la trouvait ;
- le plafond borne la **frontière** seule : `g` et `venu_de` ne sont jamais vidés, donc la mémoire totale mesurée en section 2 ne redescend pas sous le budget. Un plafond de frontière est un plafond de file, pas un plafond de mémoire.

In [5]:
def astar_frontiere_bornee(depart, budget):
    """A* a frontiere plafonnee : au-dela de `budget`, on jette le pire f.

    Rend (cout, expansions, jetes, chemin). `jetes` compte les etats retires.
    """
    g = {depart: 0}
    venu_de = {}
    compteur = itertools.count()
    tas = [(manhattan(depart), next(compteur), depart)]
    open_set = {depart}
    expansions = jetes = 0
    while tas:
        _, _, courant = heapq.heappop(tas)
        if courant not in open_set:
            continue
        open_set.discard(courant)
        expansions += 1
        if courant == BUT:
            cout = g[courant]
            chemin = [courant]
            while courant in venu_de:
                courant = venu_de[courant]
                chemin.append(courant)
            return cout, expansions, jetes, chemin[::-1]
        for suivant in successeurs(courant):
            ng = g[courant] + 1
            if ng < g.get(suivant, INFINI):
                g[suivant] = ng
                venu_de[suivant] = courant
                heapq.heappush(tas, (ng + manhattan(suivant), next(compteur), suivant))
                open_set.add(suivant)
        while len(open_set) > budget:
            victime = max(open_set, key=lambda e: g[e] + manhattan(e))
            open_set.discard(victime)
            jetes += 1
    return INFINI, expansions, jetes, None

### Ce que le plafond promet, avant qu'on le mesure

La fonction est déterministe et son contrat est simple : tant que le budget reste au-dessus du pic de frontière de l'instance, elle ne jette **aucun** état et se comporte exactement comme A\* — même coût, même garantie. C'est ce qui autorise un témoin positif : si un budget franc rendait un coût différent de l'optimum, c'est l'instrument qu'il faudrait mettre en cause, pas l'algorithme. Le tableau qui suit chiffre ce point, budget par budget.


In [6]:
BUDGETS = (8, 16, 32, 64, 128)
optimum = {graine: r[0] for graine, r in resultats_astar.items()}

print("cout rendu par budget et par instance (optimum entre parentheses)")
print(f"{'budget':>6} | {'graine 1 (20)':>14} | {'graine 2 (14)':>14} | {'graine 3 (18)':>14}")
print("-" * 58)
for budget in BUDGETS:
    ligne = []
    for graine, etat in INSTANCES.items():
        cout, expansions, jetes, chemin = astar_frontiere_bornee(etat, budget)
        rendu = "echec" if chemin is None else str(cout)
        ligne.append(f"{rendu:>14}")
    print(f"{budget:>6} | " + " | ".join(ligne))

print()
print("etats jetes, cumules sur l'execution")
print(f"{'budget':>6} | {'graine 1':>10} | {'graine 2':>10} | {'graine 3':>10}")
print("-" * 44)
for budget in BUDGETS:
    ligne = []
    for graine, etat in INSTANCES.items():
        _, _, jetes, _ = astar_frontiere_bornee(etat, budget)
        ligne.append(f"{jetes:>10}")
    print(f"{budget:>6} | " + " | ".join(ligne))

cout rendu par budget et par instance (optimum entre parentheses)
budget |  graine 1 (20) |  graine 2 (14) |  graine 3 (18)
----------------------------------------------------------
     8 |             20 |             14 |             60
    16 |             20 |             14 |             44


    32 |             20 |             14 |             34


    64 |             20 |             14 |             18
   128 |             20 |             14 |             18

etats jetes, cumules sur l'execution
budget |   graine 1 |   graine 2 |   graine 3
--------------------------------------------
     8 |         20 |         24 |        264


    16 |         12 |         23 |        348
    32 |          0 |          7 |        493
    64 |          0 |          0 |         99
   128 |          0 |          0 |         42


### Lecture du résultat

Le tableau porte les deux témoins sur la **même instance** :

- **témoin positif** — graine 3, budget 64 : coût **18**, exactement l'optimum d'A\*. Le plafond n'a rien cassé ; il a jeté 99 états sans conséquence sur le résultat.
- **témoin négatif** — graine 3, budget 32 : coût **34**, soit **+89 %** sur l'optimum. À budget 16 : **44** (+144 %). À budget 8 : **60** (+233 %).

Et le détail qui rend le défaut intéressant : sur les graines 1 et 2, même le budget 8 rend l'optimum. Le plafond naïf ne casse pas **toujours** l'optimalité — il la casse quand le budget passe sous ce que l'instance demande, et rien dans l'exécution ne signale qu'on est passé dessous. Une instance facile valide le raccourci ; c'est l'instance difficile qui le dément.

Notez aussi le nombre d'états jetés : 493 à budget 32 sur la graine 3, 99 à budget 64. Un état jeté revient par une autre route — jeter coûte donc du travail en plus du coût en qualité.

## 5. Synthèse : trois réponses à « mémoire bornée »

| Stratégie | Mémoire mesurée | Optimalité | Coût réel |
|---|---|---|---|
| **A\*** (graphe) | états atteints : 77 / 88 / **442** (dont un pic de frontière de 28 / 39 / 170) | garantie | référence |
| **IDA\*** | pic de pile : 21 / 15 / **19** | garantie | revisites entre bornes |
| **A\* plafonné** | frontière : **budget fixé** — le dictionnaire `g`, lui, n'est pas borné | perdue sous le seuil | + jetés, + expansions |

Le plafond et IDA\* bornent tous deux la frontière ; seul IDA\* **transporte la garantie**. La différence tient à ce qu'ils oublient : le plafond jette un état sans conserver ce qu'il valait, IDA\* oublie la frontière entière mais la **reconstruit** à chaque borne, ce qui préserve l'ordre d'exploration.

Ne pas confondre les deux colonnes de mémoire : le plafond d'A\* borne la **frontière**, pas le dictionnaire des états atteints — or c'est ce dictionnaire qui domine la mémoire totale mesurée en section 2. Un plafond de frontière ne rend donc pas le processus borné en mémoire.

Entre les deux existe une famille d'algorithmes qui garde les deux propriétés : **SMA\*** (Russell, 1992, *Efficient memory-bounded search methods*) sauvegarde, dans le parent, la plus petite valeur `f` du sous-arbre oublié — l'état jeté redevient un simple candidat à ré-explorer, et l'optimalité survit tant que la solution tient dans le budget. Ce carnet **ne l'implémente pas** : c'est l'exercice 3 qui en pose la question, et une implémentation se mesure avant d'être enseignée.

## Exercice 1 — Mesurer le pic d'une frontière

Écrivez une fonction `pic_de_frontiere(etat)` qui rejoue A\* sur `etat` et rend le **plus grand nombre d'états distincts** simultanément en attente.

Contraintes :

1. ne modifiez pas `astar` : écrivez une variante, ou réutilisez la mesure déjà rendue par `astar` ;
2. le résultat doit être un entier, et non le nombre d'entrées du tas (les doublons périmés ne sont pas de la mémoire) ;
3. vérifiez sur les trois instances : vous devez retrouver 28, 39 et 170.

In [7]:
def pic_de_frontiere(etat):
    # TODO etudiant : rendre le pic de frontiere d'A* sur `etat`.
    # Indice : `astar` rend deja ce pic en troisieme position.
    pass


# Etape 1 : appelez pic_de_frontiere sur chaque instance et affichez le resultat.
# Etape 2 : comparez a la troisieme colonne du tableau de la section 2.
print("Exercice 1 a completer")

Exercice 1 a completer


## Exercice 2 — Trouver le budget critique

Pour l'instance de la graine 3, cherchez par **dichotomie** le plus petit budget `B` tel que `astar_frontiere_bornee(etat, B)` rende encore l'optimum 18.

La mesure est falsifiable : le budget trouvé doit rendre 18, et `B - 1` doit rendre strictement plus que 18 (ou échouer). Si votre dichotomie rend un budget qui ne satisfait pas ces deux propriétés, c'est la dichotomie qui est fausse, pas le tableau.

Attention : la propriété « le coût est optimal à partir d'un certain budget » n'est pas démontrée dans ce carnet — elle est **observée** sur cette instance. Une dichotomie suppose cette monotonie ; dites dans votre réponse si vos mesures la confirment.

In [8]:
def budget_critique(etat, optimum):
    # TODO etudiant : dichotomie sur le budget, entre 1 et le pic de frontiere.
    # Indice : `astar_frontiere_bornee` rend (cout, expansions, jetes, chemin).
    pass


# Etape 1 : appelez budget_critique sur INSTANCES[3] avec optimum=18.
# Etape 2 : verifiez que le budget rend 18 et que budget - 1 rend autre chose.
print("Exercice 2 a completer")

Exercice 2 a completer


## Exercice 3 — Un plafond qui ne perd pas la garantie ?

Le plafond naïf jette l'état de plus grand `f` et **oublie ce qu'il valait**. SMA\* propose de s'en souvenir : quand on jette un état, on remonte dans son parent la plus petite valeur `f` du sous-arbre perdu, et le parent redevient un candidat à ré-explorer.

Décrivez, en trois à cinq phrases, **ce que cette sauvegarde change** dans le comportement de l'algorithme, et pourquoi elle suffit à préserver l'optimalité tant que la solution tient en mémoire. Vous pouvez vous appuyer sur le tableau de la section 4 : à budget 32, que faudrait-il que l'algorithme se rappelle pour ne pas rendre 34 ?

Ne codez pas SMA\* ici. Une implémentation fausse enseignée vaut moins qu'une question bien posée.

In [9]:
# TODO etudiant : votre reponse en commentaire, ou dans une chaine affichee.
reponse_exercice_3 = ""


def verifier_longueur(texte):
    """Controle minimal : une reponse de trois a cinq phrases."""
    phrases = [p for p in texte.replace("!", ".").replace("?", ".").split(".") if p.strip()]
    return len(phrases)


print("Exercice 3 a completer :", verifier_longueur(reponse_exercice_3), "phrase(s) detectee(s)")

Exercice 3 a completer : 0 phrase(s) detectee(s)


## Ce qu'il faut retenir

1. **La mémoire d'A\* n'est pas une constante** : sur un même domaine et à travail comparable, le pic de frontière varie d'un facteur 6 (28 contre 170) et les états atteints d'un facteur 6 eux aussi (77 contre 442). « Ça tient en mémoire » se mesure — et c'est le dictionnaire jamais vidé, 2,3 à 2,8 fois la frontière, qui domine.
2. **IDA\* échange la frontière contre la pile** : mémoire `O(d)`, coût identique sur les trois instances, et 19 états contre 442 sur la plus difficile.
3. **Un plafond de frontière ne transporte pas la garantie** : il la perd silencieusement dès que le budget passe sous le besoin de l'instance — 34 et 60 au lieu de 18 — et les instances faciles ne le révèlent pas.
4. **La différence est dans ce qu'on oublie** : jeter sans mémoriser casse l'ordre d'exploration ; SMA\* mémorise la valeur du sous-arbre perdu et le récupère.

### Références

- S. Russell, *Efficient memory-bounded search methods*, ECAI-92 — l'algorithme SMA\* nommé en section 5.
- S. Russell & P. Norvig, *Artificial Intelligence: A Modern Approach*, §3.5 — recherche à mémoire bornée (IDA\*, SMA\*).
- R. Korf, *Depth-first iterative-deepening: an optimal admissible tree search*, Artificial Intelligence 27(1), 1985 — IDA\*.
- [Search-03 (recherche informée et A\*)](Search-03-Informed.ipynb) — le carnet parent, qui mesure les expansions et non la mémoire.